In [4]:
# Core libraries
import pandas as pd
import numpy as np
import re
from datetime import datetime

# The star of the show
from google_play_scraper import app, reviews, Sort

print("Libraries loaded successfully!")

Libraries loaded successfully!


In [ ]:
# Step 1: Get app metadata (rating, installs, description...) for CBE, BOA, Dashen banks

# Map clean display names to their respective Play Store App IDs

banks = {
    "Commercial Bank of Ethiopia": "com.combanketh.mobilebanking",
    "Bank of Abyssinia": "com.boa.boaMobileBanking",
    "Dashen Bank": "com.dashen.dashensuperapp"
}

# Iteratively fetch and display app metadata
for bank_name, app_id in banks.items():
    try:
    # Fetch data directly inside the loop using the current app_id
        app_info = app(app_id, lang='en', country='et')     # Language: English, Country: Ethiopia
    
        print("=" * 50)
        print(f"{bank_name} App Info")
        print("=" * 50)
        print(f"App Title    : {app_info['title']}")
        print(f"Current Score: {app_info['score']}")
        print(f"Total Ratings: {app_info['ratings']:,}")
        print(f"Total Reviews: {app_info['reviews']:,}")
        print(f"Installs     : {app_info['installs']}\n")

    except Exception as e:
        # If an ID fails (like a 404), catch it here and keep going
        print("=" * 50)
        print(f"⚠️ Error loading {bank_name}")
        print("=" * 50)
        print(f"Could not retrieve App ID: '{app_id}'")
        print(f"Details: {e}\n")

Commercial Bank of Ethiopia App Info
App Title    : Commercial Bank of Ethiopia
Current Score: 4.2891264
Total Ratings: 48,382
Total Reviews: 9,316
Installs     : 5,000,000+

Bank of Abyssinia App Info
App Title    : BoA Mobile
Current Score: 4.3876925
Total Ratings: 9,234
Total Reviews: 1,462
Installs     : 1,000,000+

Dashen Bank App Info
App Title    : Dashen Bank
Current Score: 4.258407
Total Ratings: 5,643
Total Reviews: 1,023
Installs     : 1,000,000+



In [ ]:
# Step 2: Scrape reviews

# Dictionary to map clean display names to their App IDs
banks = {
    "Commercial Bank of Ethiopia": "com.combanketh.mobilebanking",
    "Bank of Abyssinia": "com.boa.boaMobileBanking",
    "Dashen Bank": "com.dashen.dashensuperapp"
}

# Master dictionary to store the actual raw reviews lists for each bank
all_bank_reviews = {}

# 1. Iterate through each bank to scrape reviews
for bank_name, app_id in banks.items():  
    try:
        # Scrape reviews
        result, continuation_token = reviews(
            app_id,
            lang='en',
            country='et',
            sort=Sort.NEWEST,       # Most recent first
            count=500,              # Target count
            filter_score_with=None  # All star ratings
        )
        
        # Store the list of reviews using the bank's name as the key
        all_bank_reviews[bank_name] = result
        print(f"✅ Success: Collected {len(result)} raw reviews for {bank_name}\n")
        
    except Exception as e:
        # **Safety Net:** If one bank fails (e.g., a temporary network glitch or an ID change), it catches errors and keep the loop running
        print(f"❌ Error scraping {bank_name} due to error: {e}\n")

# --- FINAL VALIDATION CHECK ---
print("=" * 50)
print("FINAL COLLECTION CHECK SUMMARY")
print("=" * 50)
for bank, status in scraped_data_summary.items():
    print(f"{bank:<30} : {status}")
print("=" * 50)

# 2. Inspect a single raw review from each bank
print("=" * 60)
print("INSPECTING A SAMPLE REVIEW FOR EACH BANK")
print("=" * 60)

# Iterate through all banks in the collected data
for bank_name, reviews_list in all_bank_reviews.items():
    print(f"\nTarget Bank: {bank_name}")
    print("-" * 50)
    
    if reviews_list and len(reviews_list) > 0:
        # Grab the very first review dictionary from the current bank's list
        sample_review = reviews_list[0]
        
        # Display all the available keys safely
        print(f"Keys available in this review: {list(sample_review.keys())}\n")
        print("First raw review data details:")
        
        for key, value in sample_review.items():
            print(f" {key:<20}: {value}")
            
    else:
        print(f"No sample data available for {bank_name}. Check your collection step.")
    print("-" * 50)

✅ Success: Collected 500 raw reviews for Commercial Bank of Ethiopia

✅ Success: Collected 500 raw reviews for Bank of Abyssinia

✅ Success: Collected 500 raw reviews for Dashen Bank

FINAL COLLECTION CHECK SUMMARY
Commercial Bank of Ethiopia    : 500
Bank of Abyssinia              : 500
Dashen Bank                    : 500
INSPECTING A SAMPLE REVIEW FOR EACH BANK

Target Bank: Commercial Bank of Ethiopia
--------------------------------------------------
Keys available in this review: ['reviewId', 'userName', 'userImage', 'content', 'score', 'thumbsUpCount', 'reviewCreatedVersion', 'at', 'replyContent', 'repliedAt', 'appVersion']

First raw review data details:
  reviewId              : ba0c5d66-8085-4bff-908b-f553c7b14ff5
  userName              : Yalew Mamed
  userImage             : https://play-lh.googleusercontent.com/a/ACg8ocIJXtC-Q6pj9HbzXLNKgIBuWYQD_rm08H-GXwurQhvLOC88Yg=mo
  content               : It's not allowing me to transfer money.
  score                 : 2
  thumbsUp

In [ ]:
# Define the core fields we expect to find, using .get() to handle missing ones
        fields_to_check = [
            'reviewId', 'userName', 'userImage', 'content', 'score', 
            'thumbsUpCount', 'reviewCreatedVersion', 'at', 
            'replyContent', 'repliedAt', 'appVersion'
        ]
        
        for field in fields_to_check:
            # .get(field, "Not Provided") returns the value if it exists, or the fallback text if missing
            value = sample_review.get(field, "⚠️ Field Missing")
            print(f"  {field:<22}: {value}")